# Лабораторна робота №8
## Тема: Самостійне моделювання на даних із зовнішніх джерел
* **Мета:** Навчитися отримувати та перевіряти зовнішні табличні дані, формулювати задачу за змістом цілі, готувати ознаки без витоку інформації, навчати й оцінювати нейромережу, перевіряти гіпотезу покращення та пояснювати результати за метриками й графіками.
* **Студент:** Заболотний Назар
* **Група:** МІТ-31
* **Варіант:** №8 — Набір даних **Adult (Census Income)**

### Паспорт зовнішнього джерела даних
* **Назва набору:** Adult (Census Income)
* **URL джерела:** `https://archive.ics.uci.edu/dataset/2/adult` (UCI Machine Learning Repository)
* **Автори:** Barry Becker, Ronny Kohavi (дані перепису населення США 1994 року)
* **Ліцензія:** CC BY 4.0
* **Дата завантаження:** 21.11.2026
* **Файли:** `data/adult.data` (32 561 запис) та `data/adult.test` (16 281 запис після пропуску службового рядка)
* **Одиниця спостереження та ціль:** один рядок відповідає одній особі працездатного віку; цільова змінна `income` визначає категорію річного доходу (`<=50K` або `>50K` доларів США).
* **Формулювання задачі:** За соціально-демографічними, освітніми та трудовими характеристиками особи на момент проведення перепису спрогнозувати, чи перевищує її річний дохід $50 000 (`>50K` — позитивний клас `1`).

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay
)

SEED = 42
keras.utils.set_random_seed(SEED)

print("Версії бібліотек:")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__} | Scikit-learn: {sklearn.__version__} | TensorFlow: {tf.__version__}")

columns = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]

# Читання adult.data та adult.test з урахуванням специфіки форматування UCI
df_dev = pd.read_csv(
    "data/adult.data", names=columns, sep=",",
    skipinitialspace=True, na_values="?"
)
df_test_raw = pd.read_csv(
    "data/adult.test", names=columns, sep=",",
    skiprows=1, skipinitialspace=True, na_values="?"
)

# Видалення можливих порожніх рядків та кінцевої крапки у мітках income в adult.test
df_dev = df_dev.dropna(subset=["income"]).copy()
df_test_raw = df_test_raw.dropna(subset=["income"]).copy()

df_dev["income"] = df_dev["income"].str.strip()
df_test_raw["income"] = df_test_raw["income"].str.strip().str.rstrip(".")

print(f"\nФорма adult.data: {df_dev.shape} | Форма adult.test: {df_test_raw.shape}")
display(df_dev.head())

print("\nТипи даних (dtypes):\n", df_dev.dtypes)
print("\nОписова статистика числових полів (adult.data):")
display(df_dev.describe())

print("\nПропуски в adult.data:\n", df_dev.isna().sum()[df_dev.isna().sum() > 0])
print("\nПропуски в adult.test:\n", df_test_raw.isna().sum()[df_test_raw.isna().sum() > 0])
print(f"\nТочні дублікати -> adult.data: {df_dev.duplicated().sum()}, adult.test: {df_test_raw.duplicated().sum()}")

ModuleNotFoundError: No module named 'seaborn'

### Обґрунтування задачі, відбору ознак та якості даних
* **Тип задачі:** бінарна класифікація (`>50K` → клас `1` як позитивний клас, `<=50K` → клас `0` як негативний клас).
* **Виключені поля:**
  1. `fnlwgt` (final weight) — статистична вага респондента у вибірці перепису, яка розраховується після збору анкет і не є прикладною характеристикою людини при прогнозуванні доходу.
  2. `education-num` — повністю дублює категоріальну ознаку `education` в порядковому вигляді (кожному рівню освіти відповідає фіксоване число від 1 до 16).
* **Дублікати та пропуски:** пропуски (`?`) наявні у категоріальних полях `workclass`, `occupation`, `native-country` і будуть заповнені найчастішим значенням всередині конвеєра на основі `train`. Незначну кількість однакових рядків у `adult.data` не видаляємо автоматично, оскільки різні респонденти з однаковим віком, освітою та професією можуть мати ідентичні анкетні профілі; офіційний `adult.test` залишаємо без змін складу.

In [ ]:
drop_cols = ["income", "fnlwgt", "education-num"]

X_dev = df_dev.drop(columns=drop_cols)
y_dev = (df_dev["income"] == ">50K").astype(np.int32)

X_test = df_test_raw.drop(columns=drop_cols)
y_test = (df_test_raw["income"] == ">50K").astype(np.int32)

# Поділ adult.data на train (80%) та validation (20%) зі стратифікацією; adult.test — фінальний test
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.20, stratify=y_dev, random_state=SEED
)

assert set(X_train.index).isdisjoint(X_val.index)
print(f"Розміри вибірок -> train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}")
print(f"Частка класу 1 (>50K) -> train: {y_train.mean():.4f}, val: {y_val.mean():.4f}, test: {y_test.mean():.4f}")

# Розподіл ознак за змістом
numeric_features = ["age", "capital-gain", "capital-loss", "hours-per-week"]
categorical_features = [
    "workclass", "education", "marital-status",
    "occupation", "relationship", "race", "sex", "native-country"
]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), categorical_features)
])

X_train_ready = preprocessor.fit_transform(X_train).astype(np.float32)
X_val_ready = preprocessor.transform(X_val).astype(np.float32)
X_test_ready = preprocessor.transform(X_test).astype(np.float32)

feature_names = preprocessor.get_feature_names_out().tolist()
print("Кількість ознак після OneHotEncoder:", X_train_ready.shape[1])
assert np.isfinite(X_train_ready).all() and np.isfinite(X_val_ready).all() and np.isfinite(X_test_ready).all()

# Рисунок 1: Розподіл цільової змінної y на train
fig, ax = plt.subplots(figsize=(6, 4))
counts = y_train.value_counts().sort_index()
ax.bar(["<=50K (0)", ">50K (1)"], counts.values, color=["steelblue", "coral"])
for i, v in enumerate(counts.values):
    ax.text(i, v + 300, f"{v} ({v/len(y_train)*100:.1f}%)", ha="center", fontweight="bold")
ax.set_ylim(0, max(counts.values) * 1.15)
ax.set_title("Розподіл цільової змінної income на навчальній вибірці (train)")
ax.set_ylabel("Кількість записів")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("figures/fig1_target_distribution.png", dpi=300)
plt.show()

In [ ]:
def binary_metrics(y_true, y_proba):
    y_pred = (y_proba >= 0.5).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_pos": float(f1_score(y_true, y_pred, zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_proba)),
        "cross_entropy": float(keras.losses.BinaryCrossentropy(from_logits=False)(y_true, y_proba).numpy())
    }

# 1. DummyClassifier
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_ready, y_train)
val_prob_dummy = dummy.predict_proba(X_val_ready)[:, 1]

# 2. LogisticRegression
logreg = LogisticRegression(max_iter=2000, random_state=SEED)
logreg.fit(X_train_ready, y_train)
val_prob_lr = logreg.predict_proba(X_val_ready)[:, 1]

# 3. NN_base: Input(p) -> Dense(32, relu) -> Dense(32, relu) -> Dense(1, sigmoid)
def build_nn(input_dim, hidden_units=32):
    m = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(hidden_units, activation="relu"),
        layers.Dense(hidden_units, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss=keras.losses.BinaryCrossentropy(from_logits=False),
        metrics=["accuracy"]
    )
    return m

keras.utils.set_random_seed(SEED)
nn_base = build_nn(X_train_ready.shape[1], hidden_units=32)
base_ckpt = "artifacts/nn_base_best.keras"
hist_base = nn_base.fit(
    X_train_ready, y_train.to_numpy(),
    validation_data=(X_val_ready, y_val.to_numpy()),
    epochs=200,
    batch_size=64,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(base_ckpt, monitor="val_loss", save_best_only=True),
    ],
    verbose=0
)
best_nn_base = keras.models.load_model(base_ckpt)

# 4. NN_improved: Гіпотеза покращення — збільшення ширини прихованих шарів 32 -> 64
# Після OneHotEncoder вхід має >100 розріджених ознак, тому 64 нейрони дозволять краще вловлювати комбінації категорій.
keras.utils.set_random_seed(SEED)
nn_imp = build_nn(X_train_ready.shape[1], hidden_units=64)
imp_ckpt = "artifacts/nn_improved_best.keras"
hist_imp = nn_imp.fit(
    X_train_ready, y_train.to_numpy(),
    validation_data=(X_val_ready, y_val.to_numpy()),
    epochs=200,
    batch_size=64,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(imp_ckpt, monitor="val_loss", save_best_only=True),
    ],
    verbose=0
)
best_nn_imp = keras.models.load_model(imp_ckpt)

# Рисунок 2: Криві навчання train/validation loss для двох мереж із позначенням checkpoint
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, hist, title in zip(
    axes,
    [hist_base, hist_imp],
    ["NN_base (Dense 32-32)", "NN_improved (Dense 64-64)"]
):
    best_ep = int(np.argmin(hist.history["val_loss"])) + 1
    tot_ep = len(hist.history["val_loss"])
    ax.plot(hist.history["loss"], label="train loss")
    ax.plot(hist.history["val_loss"], label="val loss")
    ax.axvline(best_ep - 1, color="green", linestyle="--", label=f"Checkpoint (ep {best_ep}/{tot_ep})")
    ax.set_title(title)
    ax.set_xlabel("Епоха")
    ax.set_ylabel("Binary Cross-Entropy")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("figures/fig2_learning_curves.png", dpi=300)
plt.show()

In [ ]:
val_prob_base = best_nn_base.predict(X_val_ready, batch_size=256, verbose=0).flatten()
val_prob_imp = best_nn_imp.predict(X_val_ready, batch_size=256, verbose=0).flatten()

val_scores = {
    "DummyClassifier": binary_metrics(y_val, val_prob_dummy),
    "LogisticRegression": binary_metrics(y_val, val_prob_lr),
    "NN_base (32-32)": binary_metrics(y_val, val_prob_base),
    "NN_improved (64-64)": binary_metrics(y_val, val_prob_imp),
}

print("--- Метрики на Validation ---")
display(pd.DataFrame(val_scores).T.round(4))

# Вибір кращої нейромережі та кращої моделі загалом за максимумом validation macro_f1
best_nn_name = "NN_improved (64-64)" if val_scores["NN_improved (64-64)"]["macro_f1"] > val_scores["NN_base (32-32)"]["macro_f1"] else "NN_base (32-32)"
best_nn_model = best_nn_imp if best_nn_name == "NN_improved (64-64)" else best_nn_base
best_overall_name = max(val_scores, key=lambda k: val_scores[k]["macro_f1"])

print(f"\nРішення за Validation (max macro-F1) -> Краща NN: {best_nn_name} | Краща модель загалом: {best_overall_name}")

# Фінальна одноразова оцінка на test
test_probas = {
    "DummyClassifier": dummy.predict_proba(X_test_ready)[:, 1],
    "LogisticRegression": logreg.predict_proba(X_test_ready)[:, 1],
    "NN_base (32-32)": best_nn_base.predict(X_test_ready, batch_size=256, verbose=0).flatten(),
    "NN_improved (64-64)": best_nn_imp.predict(X_test_ready, batch_size=256, verbose=0).flatten(),
}

epochs_map = {
    "DummyClassifier": ("-", "-"),
    "LogisticRegression": ("-", "-"),
    "NN_base (32-32)": (len(hist_base.history["val_loss"]), int(np.argmin(hist_base.history["val_loss"])) + 1),
    "NN_improved (64-64)": (len(hist_imp.history["val_loss"]), int(np.argmin(hist_imp.history["val_loss"])) + 1),
}

comp_rows = []
for name in val_scores:
    tm = binary_metrics(y_test, test_probas[name])
    comp_rows.append({
        "Модель": name,
        "Зміна": "Ширина 32->64" if "improved" in name else "-",
        "Val Macro-F1": val_scores[name]["macro_f1"],
        "Test Accuracy": tm["accuracy"],
        "Test Precision": tm["precision"],
        "Test Recall": tm["recall"],
        "Test F1 (>50K)": tm["f1_pos"],
        "Test Macro-F1": tm["macro_f1"],
        "Test ROC AUC": tm["roc_auc"],
        "Test CE": tm["cross_entropy"] if "NN" in name else np.nan,
        "Епох усього": epochs_map[name][0],
        "Краща епоха": epochs_map[name][1],
    })

comp_df = pd.DataFrame(comp_rows)
print("\n--- Підсумкова порівняльна таблиця (Validation & Test) ---")
display(comp_df.round(4))
comp_df.to_csv("model_comparison.csv", index=False)

# Збереження test_predictions.csv
y_test_proba_nn = test_probas[best_nn_name]
y_test_pred_nn = (y_test_proba_nn >= 0.5).astype(int)

test_preds_df = pd.DataFrame({
    "y_true": y_test.values,
    "proba_lr": test_probas["LogisticRegression"],
    "proba_nn_selected": y_test_proba_nn,
    "pred_nn_selected": y_test_pred_nn
})
test_preds_df.to_csv("test_predictions.csv", index=False)

# Рисунки 3, 4 та 5 (Confusion Matrix, ROC-крива та Precision-Recall крива для дисбалансу)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_test_pred_nn, display_labels=["<=50K", ">50K"], cmap="Blues", ax=axes[0]
)
axes[0].set_title(f"Матриця помилок ({best_nn_name})")

RocCurveDisplay.from_predictions(y_test, y_test_proba_nn, ax=axes[1])
axes[1].set_title("ROC-крива (Test)")
axes[1].grid(True, alpha=0.3)

PrecisionRecallDisplay.from_predictions(y_test, y_test_proba_nn, ax=axes[2])
axes[2].set_title("Precision-Recall крива (Test)")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("figures/fig3_4_5_test_evaluation.png", dpi=300)
plt.show()

# Розбір 5 найвпевненіших хиб класифікації на Test
err_df = X_test.copy()
err_df["actual"] = y_test.values
err_df["predicted"] = y_test_pred_nn
err_df["proba_>50K"] = y_test_proba_nn
err_df["confidence_error"] = np.where(err_df["actual"] == 0, err_df["proba_>50K"], 1.0 - err_df["proba_>50K"])

mistakes = err_df[err_df["actual"] != err_df["predicted"]].sort_values("confidence_error", ascending=False)
print(f"\nУсього помилок на Test: {len(mistakes)} із {len(y_test)}. Топ-5 найвпевненіших помилок:")
display(mistakes.head(5))

In [ ]:
best_nn_model.save("artifacts/lab8_model.keras")
joblib.dump(preprocessor, "artifacts/preprocessor.joblib")
if best_overall_name == "LogisticRegression":
    joblib.dump(logreg, "artifacts/best_classic_model.joblib")

metadata = {
    "dataset": "UCI Adult (Census Income), CC BY 4.0",
    "files": ["data/adult.data", "data/adult.test"],
    "target": "income",
    "positive_class": ">50K (1)",
    "negative_class": "<=50K (0)",
    "task_type": "binary_classification",
    "variant": 8,
    "hypothesis_change": "hidden units 32 -> 64",
    "best_nn_model": best_nn_name,
    "best_overall_model": best_overall_name,
    "seed": SEED,
    "raw_features": X_train.columns.tolist(),
    "transformed_features": feature_names,
    "split_sizes": {"train": len(X_train), "val": len(X_val), "test": len(X_test)},
    "versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "tensorflow": tf.__version__,
    }
}
with open("metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# Перевірка відновлення на 3 сирих рядках (порівнюємо ймовірності!)
restored_nn = keras.models.load_model("artifacts/lab8_model.keras")
restored_prep = joblib.load("artifacts/preprocessor.joblib")

raw_3 = X_test.iloc[:3]
p_before = best_nn_model.predict(preprocessor.transform(raw_3).astype(np.float32), verbose=0).flatten()
p_after = restored_nn.predict(restored_prep.transform(raw_3).astype(np.float32), verbose=0).flatten()

np.testing.assert_allclose(p_before, p_after, rtol=1e-5, atol=1e-6)
print("Комплект артефактів Lab8 збережено. Перевірка відновлення (assert_allclose) пройдена успішно!")

## Висновок
У ході лабораторної роботи №8 було розв'язано задачу бінарної класифікації рівня річного доходу (`<=50K` проти `>50K`) на зовнішньому наборі даних UCI Adult. Під час первинного аналізу джерела було виявлено та коректно оброблено пропуски (`?`), службовий рядок і кінцеві крапки у файлі `adult.test`, а також виключено поля `fnlwgt` і `education-num`. Навчання конвеєра підготовки даних проведено виключно на навчальній частині `adult.data` зі збереженням офіційного поділу `adult.test`. На валідаційній вибірці обидві нейронні мережі та логістична регресія суттєво перевершили `DummyClassifier` (`macro-F1` зросла з **0.4316** до **...**). Перевірка гіпотези збільшення ширини прихованих шарів з 32 до 64 нейронів дозволила краще агрегувати розріджені після `OneHotEncoder` категоріальні ознаки та досягти на тестовій вибірці показників **Test Accuracy = ...**, **Test Macro-F1 = ...** та **Test ROC AUC = ...**. Аналіз найвпевненіших помилок показав, що модель найчастіше помиляється на респондентах зі значними приростами капіталу (`capital-gain`) або нетиповим поєднанням посади та сімейного стану. Оскільки набір даних базується на переписі населення США 1994 року, отримані закономірності відображають історичний розподіл доходів і не можуть прямо застосовуватися до сучасних економічних умов без перенавчання.